#### 다음 실습 코드는 학습 목적으로만 사용 바랍니다. 문의 : audit@korea.ac.kr 임성열 Ph.D.

In [ ]:
# python -m venv llm
# pip install -r requirements-llm.txt
# 이미 설치했다면 아래 셀은 실행하지 않습니다.
# ------------------------------------------------------------

# %pip install -U "crewai[openai]>=1.15,<2.0" python-dotenv


In [1]:
# 커널 재시작 후 가장 먼저 실행
from dotenv import load_dotenv
load_dotenv(override=True)

import os

api_key = os.getenv("OPENAI_API_KEY")
print(api_key[:12], api_key[-4:])

sk-proj-XMw6 uscA


In [2]:
# 경고 메시지 정리
import warnings
warnings.filterwarnings("ignore")


In [3]:
from crewai import Agent, Crew, LLM, Process, Task


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 현재 작업 폴더 또는 상위 폴더에서 .env 파일을 탐색합니다.
def find_env_file(start: Path) -> Path | None:
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    return None

env_path = find_env_file(Path.cwd())
if env_path is None:
    raise FileNotFoundError(
        ".env 파일을 찾을 수 없습니다. 노트북과 같은 폴더에 .env 파일을 만들고 "
        "OPENAI_API_KEY=... 형식으로 API 키를 저장하세요."
    )

load_dotenv(dotenv_path=env_path, override=False)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 OPENAI_API_KEY가 설정되어 있지 않습니다.")

# CrewAI가 사용할 OpenAI 모델을 명시적으로 지정합니다.
# 필요하면 .env의 OPENAI_MODEL_NAME 값만 바꾸면 됩니다.
# 환경변수 OPENAI_MODEL_NAME을 읽고, 없으면 "openai/gpt-4o-mini"를 사용
model_name = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini") 
if not model_name.startswith("openai/"):
    model_name = f"openai/{model_name}"

llm = LLM(
    model=model_name,
    api_key=api_key,
    temperature=0.3,
)

print(f".env 로드 완료: {env_path}")
print(f"사용 모델: {model_name}")


.env 로드 완료: /Users/phoenix/Eagle/2026_LLM/code/.env
사용 모델: openai/gpt-4o-mini


### `.env` 파일 예시

노트북과 같은 폴더에 `.env` 파일을 만들고 아래와 같이 작성합니다.

```dotenv
OPENAI_API_KEY=sk-여기에_실제_API_KEY
OPENAI_MODEL_NAME=openai/gpt-4o-mini
```

`.env` 파일은 Git 저장소에 올리지 않도록 `.gitignore`에 추가하세요.


Agent(에이전트) 생성하기

Agent를 정의하고, role(역할), goal(목표), backstory(배경 설명)를 제공합니다.

LLM(대규모 언어 모델)은 롤플레잉을 할 때 더 나은 성능을 보이는 것으로 확인되었습니다.

### Agent 정의

각 Agent에 `role`, `goal`, `backstory`, `llm`을 지정합니다.  
이 예제에서는 기획자 → 작성자 → 편집자 순서로 작업합니다.


In [ ]:
# 최신 CrewAI에서는 Task 클래스를 직접 재정의하지 않아도
# 각 Task의 output 속성과 CrewOutput을 통해 결과를 확인할 수 있습니다.

In [6]:
# 콘텐츠 기획자
planner = Agent(
    role="콘텐츠 기획자",
    goal="{topic}에 대한 흥미롭고 사실에 기반한 콘텐츠를 기획한다.",
    backstory=(
        "당신은 {topic}에 대한 블로그 글을 기획하는 전문가입니다. "
        "독자가 새로운 내용을 배우고 정보에 기반한 결정을 내릴 수 있도록 "
        "핵심 쟁점, 독자 요구, 글의 구조를 정리합니다. "
        "모든 결과물은 한국어로 작성합니다."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)


### Agent: Writer

In [7]:
# 콘텐츠 작성자
writer = Agent(
    role="콘텐츠 작성자",
    goal="{topic}에 대한 통찰력 있고 사실에 기반한 블로그 글을 작성한다.",
    backstory=(
        "당신은 콘텐츠 기획자가 제공한 기획안을 바탕으로 글을 쓰는 전문 작가입니다. "
        "객관적 사실과 개인적 해석을 구분하고, 독자가 이해하기 쉬운 구조로 작성합니다. "
        "모든 결과물은 한국어로 작성합니다."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)


### Agent: Editor

In [8]:
# 편집자
editor = Agent(
    role="편집자",
    goal="작성된 블로그 글을 명확하고 균형 잡힌 최종 원고로 편집한다.",
    backstory=(
        "당신은 콘텐츠 작성자의 원고를 검토하는 전문 편집자입니다. "
        "문법, 논리, 가독성, 균형성을 점검하고 출판 가능한 최종 글로 다듬습니다. "
        "검토 보고가 아니라 수정된 글 자체만 출력합니다. "
        "모든 결과물은 한국어로 작성합니다."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)


## Task(작업) 생성하기

- Task를 정의하고, `description`(설명), `expected_output`(예상 결과물), `agent`(수행 에이전트)를 제공합니다.


### Task: Plan

In [9]:
plan = Task(
    description=(
        "{topic}에 관한 콘텐츠 기획안을 작성하세요.\n"
        "1. 핵심 트렌드와 주요 쟁점을 정리합니다.\n"
        "2. 목표 독자와 독자의 관심사 및 어려움을 분석합니다.\n"
        "3. 서론, 핵심 섹션, 결론과 행동 유도를 포함한 상세 개요를 작성합니다.\n"
        "4. 활용할 SEO 키워드와 사실 확인이 필요한 항목을 제시합니다.\n"
        "외부 검색 도구가 없으므로 확인되지 않은 최신 뉴스나 통계를 만들어내지 마세요."
    ),
    expected_output=(
        "목표 독자 분석, 핵심 메시지, 상세 목차, SEO 키워드, "
        "사실 확인 주의사항을 포함한 한국어 콘텐츠 기획안"
    ),
    agent=planner,
)


### Task: Write

In [10]:
write = Task(
    description=(
        "앞 단계에서 작성된 콘텐츠 기획안을 바탕으로 {topic}에 관한 블로그 글을 작성하세요.\n"
        "1. SEO 키워드를 부자연스럽지 않게 포함합니다.\n"
        "2. 제목과 소제목을 명확하게 구성합니다.\n"
        "3. 서론, 본문, 결론의 흐름을 유지합니다.\n"
        "4. 검증되지 않은 수치나 사례를 사실처럼 단정하지 않습니다.\n"
        "5. 마크다운 형식으로 작성합니다."
    ),
    expected_output=(
        "제목, 소제목, 서론, 본문, 결론을 갖춘 출판 가능한 한국어 마크다운 블로그 글"
    ),
    agent=writer,
    context=[plan],
)


### Task: Edit

In [11]:
edit = Task(
    description=(
        "앞 단계의 블로그 글을 최종 편집하세요. 문법, 논리적 연결, 중복, "
        "과도한 주장, 가독성을 점검하고 자연스럽게 수정하세요. "
        "검토 완료 문장이나 편집 설명은 쓰지 말고 최종 블로그 글만 출력하세요."
    ),
    expected_output=(
        "편집 설명 없이 최종 원고만 포함한 출판 가능한 한국어 마크다운 블로그 글"
    ),
    agent=editor,
    context=[write],
)


## Crew(팀) 생성하기

- Agent들로 구성된 팀을 생성합니다
- 해당 Agent들이 수행할 작업들을 전달합니다.
    - **참고**: *이 간단한 예시에서는* 작업들이 순차적으로 수행됩니다(즉, 서로 의존적임). 따라서 목록에서의 작업 _순서_가 _중요_합니다.
- `verbose=2`를 설정하면 실행의 모든 로그를 확인할 수 있습니다.


이 설정에서:
1. `planner`가 먼저 콘텐츠를 기획합니다
2. `writer`가 기획된 내용을 바탕으로 글을 작성합니다
3. `editor`가 최종적으로 작성된 글을 검토하고 편집합니다

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    process=Process.sequential,
    verbose=True,
)

## Running the Crew

In [13]:
# Crew 실행
# API 사용량이 발생합니다.
topic = "LLM(Large Language Model)을 이용한 지능형 에이전트 경쟁력 제고 방안"
# Crew를 비동기 방식으로 실행
result = await crew.kickoff_async()

# 최종 결과 출력
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ed88069f-cb72-4d18-afad-1817e31910e0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: {topic}에 관한 콘텐츠 기획안을 작성하세요.                                                               │
│  1. 핵심 트렌드와 주요 쟁점을 정리합니다.                                                                       │
│  2. 목표 독자와 독자의 관심사 및 어려움을 분석합니다.                                                           │
│  3. 서론, 핵심 섹션, 결론과 행동 유도를 포함한 상세 개요를 작성합니다.                                          │
│  4. 활용할 SEO 키워드와 사실 확인이 필요한 항목을 제시합니다.                                                   │
│  외부 검색 도구가 없으므로 확인되지 않은 최신 뉴스나 통계를 만들어내지 마세요.                                  │
│  ID: 5700fcc3-e6a7-45c0-b8ce-4d6b3a6351c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 콘텐츠 기획자                                                                                           │
│                                                                                                                 │
│  Task: {topic}에 관한 콘텐츠 기획안을 작성하세요.                                                               │
│  1. 핵심 트렌드와 주요 쟁점을 정리합니다.                                                                       │
│  2. 목표 독자와 독자의 관심사 및 어려움을 분석합니다.                                                           │
│  3. 서론, 핵심 섹션, 결론과 행동 유도를 포함한 상세 개요를 작성합니다.                                          │
│  4. 활용할 SEO 키워드와 사실 확인이 필요한 항목을 제시합니다.                                                   │
│  외부 검색 도구가 없으므로 확인되지 않은 최신 뉴스나 통계를 만들어내지 마세요.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 콘텐츠 기획자                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 콘텐츠 기획안: {topic}에 관한 블로그 글                                                                      │
│                                                                                                                 │
│  ## 1. 핵심 트렌드와 주요 쟁점                                                                                  │
│  - **트렌드**: {topic}의 최신 동향, 기술 발전, 사회적 변화 등을 반영한 내용.                                    │
│  - **주요 쟁점**:                                                                                               │
│    - {topic}의 현재 상황과 향후 전망                                                                            │
│    - 관련 법규 및 정책 변화                                                                                     │
│    - 사회적 인식 및 문화적 변화                                                                                 │
│    - {topic}과 관련된 주요 이슈 및 논란                                                                         │
│                                                                                                                 │
│  ## 2. 목표 독자 분석                                                                                           │
│  - **목표 독자**:                                                                                               │
│    - {topic}에 관심이 있는 일반 대중                                                                            │
│    - 관련 업계 종사자 및 전문가                                                                                 │
│    - 정책 결정자 및 연구자                                                                                      │
│  - **독자의 관심사**:                                                                                           │
│    - {topic}의 최신 정보 및 동향                                                                                │
│    - 실제 사례 및 경험담                                                                                        │
│    - 문제 해결을 위한 팁과 가이드                                                                               │
│  - **독자의 어려움**:                                                                                           │
│    - 정보의 과다로 인한 선택의 어려움                                                                           │
│    - 신뢰할 수 있는 출처의 부족                                                                                 │
│    - 복잡한 용어와 개념으로 인한 이해의 어려움                                                                  │
│                                                                                                                 │
│  ## 3. 글의 구조 및 상세 개요                                                                                   │
│  ### 서론                                                                                                       │
│  - {topic}의 중요성 및 현재 상황 소개                                                                           │
│  - 독자가 이 글을 통해 얻을 수 있는 가치 설명                                                                   │
│                                                                                                                 │
│  ### 핵심 섹션                                                                                                  │
│  1. **{topic}의 정의 및 배경**                                                                                  │
│     - 기본 개념 설명                                                                               

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 편집자                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # {topic}의 최신 동향과 미래 전망                                                                              │
│                                                                                                                 │
│  ## 서론                                                                                                        │
│  {topic}은 현대 사회에서 점점 더 중요해지고 있는 주제입니다. 기술 발전과 사회적 변화가 맞물리면서 {topic}의     │
│  정의와 그 중요성이 더욱 부각되고 있습니다. 이 글을 통해 독자들은 {topic}의 기본 개념, 최신 동향, 주요 쟁점,    │
│  사례 연구, 그리고 미래 전망에 대해 깊이 있는 통찰을 얻을 수 있을 것입니다.                                     │
│                                                                                                                 │
│  ## 1. {topic}의 정의 및 배경                                                                                   │
│  {topic}은 기본적으로 [정의]입니다. 이 개념은 [역사적 배경]을 통해 발전해왔으며, 그 과정에서 여러 가지 변화와   │
│  혁신이 있었습니다. 예를 들어, {topic}의 초기 형태는 [초기 사례]와 같은 모습이었으나, 현재는 [현재의 모습]으로  │
│  진화하였습니다.                                                                                                │
│                                                                                                                 │
│  ## 2. 현재의 트렌드                                                                                            │
│  최근 {topic}의 최신 동향은 [최신 동향]입니다. 기술 발전이 이끌어내는 변화는 [기술적 발전]과 같은 형태로        │
│  나타나고 있으며, 이는 사회 전반에 걸쳐 큰 영향을 미치고 있습니다. 예를 들어, [사회적 변화]는 {topic}에 대한    │
│  인식을 변화시키고 있으며, 이는 [구체적인 사례]로 확인할 수 있습니다.                                           │
│                                                                                                                 │
│  ## 3. 주요 쟁점 및 논란                                                                                        │
│  현재 {topic}과 관련하여 논의되고 있는 주요 이슈는 [주요 이슈]입니다. 이와 관련된 다양한 시각이 존재하며,       │
│  [다양한 의견]을 통해 각기 다른 관점을 이해할 수 있습니다. 이러한 논란은 {topic}의 발전 방향에 큰 영향을        │
│  미치고 있으며, 앞으로의 정책 결정에도 중요한 요소로 작용할 것입니다.                                           │
│                                                                                                                 │
│  ## 4. 사례 연구                                                                                                │
│  성공적인 사례로는 [성공 사례]가 있으며, 이는 {topic}의 긍정적인 영향을 보여줍니다. 반면, 실패 사례로는 [실패   │
│  사례]가 있으며, 이는 {topic}을 적용하는 데 있어 주의해야 할 점을 시사합니다. 이러한 사례들은 {topic}에 대한    │
│  이해를 높이고, 향후 적용 시 유용한 교훈을 제공합니다.                                                          │
│                                                                                                                 │
│  ## 5. 미래 전망                                                                                                │
│  향후 {topic}에 예상되는 변화는 [미래 변화]입니다. 이러한 변화는 [기술적, 사회적 변화]와 밀접하게 연결되어      │
│  있으며, 독자들은 이를 준비하기 위해 [필요한 준비 사항]을 고려해야 합니다. 특히, {topic}의 발전 방향에 맞춰     │
│  개인적, 사회적 대응이 필요할 것입니다.                                                                         │
│                                                                                                                 │
│  ## 결론                                                                                                        │
│  {topic}은 현재와 미래 모두에서 중요한 주제입니다. 이 글을 통해 {topic}의 정의, 최신 동향, 주요 쟁점, 사례      │
│  연구, 그리고 미래 전망에 대해 살펴보았습니다. 독자 여러분은 이 정보를 바탕으로 {topic}에 대한 이해를 높이고,   │
│  관련된 논의에 적극적으로 참여하시기 바랍니다. 여러분의 의견과 경험을 공유해 주시면, 더 나은 논의가 이루어질    │
│  것입니다.               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 앞 단계의 블로그 글을 최종 편집하세요. 문법, 논리적 연결, 중복, 과도한 주장, 가독성을 점검하고           │
│  자연스럽게 수정하세요. 검토 완료 문장이나 편집 설명은 쓰지 말고 최종 블로그 글만 출력하세요.                   │
│  Agent: 편집자                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ed88069f-cb72-4d18-afad-1817e31910e0                                                                       │
│  Final Output: # {topic}의 최신 동향과 미래 전망                                                                │
│                                                                                                                 │
│  ## 서론                                                                                                        │
│  {topic}은 현대 사회에서 점점 더 중요해지고 있는 주제입니다. 기술 발전과 사회적 변화가 맞물리면서 {topic}의     │
│  정의와 그 중요성이 더욱 부각되고 있습니다. 이 글을 통해 독자들은 {topic}의 기본 개념, 최신 동향, 주요 쟁점,    │
│  사례 연구, 그리고 미래 전망에 대해 깊이 있는 통찰을 얻을 수 있을 것입니다.                                     │
│                                                                                                                 │
│  ## 1. {topic}의 정의 및 배경                                                                                   │
│  {topic}은 기본적으로 [정의]입니다. 이 개념은 [역사적 배경]을 통해 발전해왔으며, 그 과정에서 여러 가지 변화와   │
│  혁신이 있었습니다. 예를 들어, {topic}의 초기 형태는 [초기 사례]와 같은 모습이었으나, 현재는 [현재의 모습]으로  │
│  진화하였습니다.                                                                                                │
│                                                                                                                 │
│  ## 2. 현재의 트렌드                                                                                            │
│  최근 {topic}의 최신 동향은 [최신 동향]입니다. 기술 발전이 이끌어내는 변화는 [기술적 발전]과 같은 형태로        │
│  나타나고 있으며, 이는 사회 전반에 걸쳐 큰 영향을 미치고 있습니다. 예를 들어, [사회적 변화]는 {topic}에 대한    │
│  인식을 변화시키고 있으며, 이는 [구체적인 사례]로 확인할 수 있습니다.                                           │
│                                                                                                                 │
│  ## 3. 주요 쟁점 및 논란                                                                                        │
│  현재 {topic}과 관련하여 논의되고 있는 주요 이슈는 [주요 이슈]입니다. 이와 관련된 다양한 시각이 존재하며,       │
│  [다양한 의견]을 통해 각기 다른 관점을 이해할 수 있습니다. 이러한 논란은 {topic}의 발전 방향에 큰 영향을        │
│  미치고 있으며, 앞으로의 정책 결정에도 중요한 요소로 작용할 것입니다.                                           │
│                                                                                                                 │
│  ## 4. 사례 연구                                                                                                │
│  성공적인 사례로는 [성공 사례]가 있으며, 이는 {topic}의 긍정적인 영향을 보여줍니다. 반면, 실패 사례로는 [실패   │
│  사례]가 있으며, 이는 {topic}을 적용하는 데 있어 주의해야 할 점을 시사합니다. 이러한 사례들은 {topic}에 대한    │
│  이해를 높이고, 향후 적용 시 유용한 교훈을 제공합니다.                                                          │
│                                                                                                                 │
│  ## 5. 미래 전망                                                                                                │
│  향후 {topic}에 예상되는 변화는 [미래 변화]입니다. 이러한 변화는 [기술적, 사회적 변화]와 밀접하게 연결되어      │
│  있으며, 독자들은 이를 준비하기 위해 [필요한 준비 사항]을 고려해야 합니다. 특히, {topic}의 발전 방향에 맞춰     │
│  개인적, 사회적 대응이 필요할 것입니다.                                                                         │
│                                                                                                                 │
│  ## 결론                                                                                                        │
│  {topic}은 현재와 미래 모두에서 중요한 주제입니다. 이 글을 통해 {topic}의 정의, 최신 동향, 주요 쟁점, 사례      │
│  연구, 그리고 미래 전망에 대해 살펴보았습니다. 독자 여러분은 이 정보를 바탕으로 {topic}에 대한 이해를 높이고,   │
│  관련된 논의에 적극적으로 참여하시기 바랍니다. 여러분의 의견과 경험을 공유해 주시면, 더 나은 논의가 이루어질    │
│  것입니다.           

# {topic}의 최신 동향과 미래 전망

## 서론
{topic}은 현대 사회에서 점점 더 중요해지고 있는 주제입니다. 기술 발전과 사회적 변화가 맞물리면서 {topic}의 정의와 그 중요성이 더욱 부각되고 있습니다. 이 글을 통해 독자들은 {topic}의 기본 개념, 최신 동향, 주요 쟁점, 사례 연구, 그리고 미래 전망에 대해 깊이 있는 통찰을 얻을 수 있을 것입니다.

## 1. {topic}의 정의 및 배경
{topic}은 기본적으로 [정의]입니다. 이 개념은 [역사적 배경]을 통해 발전해왔으며, 그 과정에서 여러 가지 변화와 혁신이 있었습니다. 예를 들어, {topic}의 초기 형태는 [초기 사례]와 같은 모습이었으나, 현재는 [현재의 모습]으로 진화하였습니다.

## 2. 현재의 트렌드
최근 {topic}의 최신 동향은 [최신 동향]입니다. 기술 발전이 이끌어내는 변화는 [기술적 발전]과 같은 형태로 나타나고 있으며, 이는 사회 전반에 걸쳐 큰 영향을 미치고 있습니다. 예를 들어, [사회적 변화]는 {topic}에 대한 인식을 변화시키고 있으며, 이는 [구체적인 사례]로 확인할 수 있습니다.

## 3. 주요 쟁점 및 논란
현재 {topic}과 관련하여 논의되고 있는 주요 이슈는 [주요 이슈]입니다. 이와 관련된 다양한 시각이 존재하며, [다양한 의견]을 통해 각기 다른 관점을 이해할 수 있습니다. 이러한 논란은 {topic}의 발전 방향에 큰 영향을 미치고 있으며, 앞으로의 정책 결정에도 중요한 요소로 작용할 것입니다.

## 4. 사례 연구
성공적인 사례로는 [성공 사례]가 있으며, 이는 {topic}의 긍정적인 영향을 보여줍니다. 반면, 실패 사례로는 [실패 사례]가 있으며, 이는 {topic}을 적용하는 데 있어 주의해야 할 점을 시사합니다. 이러한 사례들은 {topic}에 대한 이해를 높이고, 향후 적용 시 유용한 교훈을 제공합니다.

## 5. 미래 전망
향후 {topic}에 예상되는 변화는 [미래 변화]입니다. 이러한 변화는 [기술적

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from IPython.display import Markdown, display

# kickoff()은 CrewOutput 객체를 반환하므로 raw 문자열을 사용합니다.
final_text = result.raw

display(Markdown(final_text))

# 단계별 결과가 필요할 때 아래 속성을 사용할 수 있습니다.
# print(plan.output.raw)
# print(write.output.raw)
# print(edit.output.raw)
